In [ ]:
!pip install -q langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="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",
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
from typing import TypedDict

class TeamState(TypedDict):
    task: str
    worker_result: str
    summary: str

In [ ]:
def worker(state: TeamState):

    answer = llm.invoke(
        "Solve this math problem, show only the final number:\n"
        + state["task"]
    ).content

    return {
        "worker_result": answer
    }

In [ ]:
def supervisor(state: TeamState):

    summary = llm.invoke(
        f"""
The worker solved:

{state['task']}

Result:

{state['worker_result']}

Write a one-line summary.
"""
    ).content

    return {
        "summary": summary
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TeamState)

builder.add_node("worker", worker)
builder.add_node("supervisor", supervisor)

builder.add_edge(START, "worker")
builder.add_edge("worker", "supervisor")
builder.add_edge("supervisor", END)

graph = builder.compile()

In [ ]:
result = graph.invoke(
    {
        "task": "What is 144 divided by 12, then plus 5?"
    }
)

print("Worker Result:")
print(result["worker_result"])

print()

print("Supervisor Summary:")
print(result["summary"])

Worker Result:
12

Supervisor Summary:
The worker solved a math problem by dividing 144 by 12, which equals 12, and then adding 5, resulting in a final answer of 17.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')